In [1]:
import kagglehub

path = kagglehub.dataset_download("maharshipandya/-spotify-tracks-dataset")

print("Path to dataset files:", path)

Path to dataset files: /Users/zzzhao/.cache/kagglehub/datasets/maharshipandya/-spotify-tracks-dataset/versions/1


In [2]:
import pandas as pd
import numpy as np
import os

path = "/Users/zzzhao/.cache/kagglehub/datasets/maharshipandya/-spotify-tracks-dataset/versions/1"

csv_file = os.path.join(path, "dataset.csv") 
df = pd.read_csv(csv_file)

print(df.head())


   Unnamed: 0                track_id                 artists  \
0           0  5SuOikwiRyPMVoIQDJUgSV             Gen Hoshino   
1           1  4qPNDBW1i3p13qLCt0Ki3A            Ben Woodward   
2           2  1iJBSr7s7jYXzM8EGcbK5b  Ingrid Michaelson;ZAYN   
3           3  6lfxq3CG4xtTiEg7opyCyx            Kina Grannis   
4           4  5vjLSffimiIP26QG5WcN2K        Chord Overstreet   

                                          album_name  \
0                                             Comedy   
1                                   Ghost (Acoustic)   
2                                     To Begin Again   
3  Crazy Rich Asians (Original Motion Picture Sou...   
4                                            Hold On   

                   track_name  popularity  duration_ms  explicit  \
0                      Comedy          73       230666     False   
1            Ghost - Acoustic          55       149610     False   
2              To Begin Again          57       210826     False   


In [3]:
df = df.drop(columns=["Unnamed: 0", "track_id", "album_name"])
df["explicit"] = df["explicit"].astype(int)
df = df.dropna()
df['tempo'] = (df['tempo'] - df['tempo'].min()) / (df['tempo'].max() - df['tempo'].min())
df['popularity'] = df['popularity'] / 100

print(df.columns)
print(df["track_genre"].unique())  


Index(['artists', 'track_name', 'popularity', 'duration_ms', 'explicit',
       'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness',
       'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo',
       'time_signature', 'track_genre'],
      dtype='object')
['acoustic' 'afrobeat' 'alt-rock' 'alternative' 'ambient' 'anime'
 'black-metal' 'bluegrass' 'blues' 'brazil' 'breakbeat' 'british'
 'cantopop' 'chicago-house' 'children' 'chill' 'classical' 'club' 'comedy'
 'country' 'dance' 'dancehall' 'death-metal' 'deep-house' 'detroit-techno'
 'disco' 'disney' 'drum-and-bass' 'dub' 'dubstep' 'edm' 'electro'
 'electronic' 'emo' 'folk' 'forro' 'french' 'funk' 'garage' 'german'
 'gospel' 'goth' 'grindcore' 'groove' 'grunge' 'guitar' 'happy'
 'hard-rock' 'hardcore' 'hardstyle' 'heavy-metal' 'hip-hop' 'honky-tonk'
 'house' 'idm' 'indian' 'indie-pop' 'indie' 'industrial' 'iranian'
 'j-dance' 'j-idol' 'j-pop' 'j-rock' 'jazz' 'k-pop' 'kids' 'latin'
 'latino' 'malay' 'mandopo

In [4]:
from sklearn.preprocessing import LabelEncoder

full_genre_list = [
    'acoustic', 'afrobeat', 'alt-rock', 'alternative', 'ambient', 'anime',
    'black-metal', 'bluegrass', 'blues', 'brazil', 'breakbeat', 'british',
    'cantopop', 'chicago-house', 'children', 'chill', 'classical', 'club',
    'comedy', 'country', 'dance', 'dancehall', 'death-metal', 'deep-house',
    'detroit-techno', 'disco', 'disney', 'drum-and-bass', 'dub', 'dubstep',
    'edm', 'electro', 'electronic', 'emo', 'folk', 'forro', 'french', 'funk',
    'garage', 'german', 'gospel', 'goth', 'grindcore', 'groove', 'grunge',
    'guitar', 'happy', 'hard-rock', 'hardcore', 'hardstyle', 'heavy-metal',
    'hip-hop', 'honky-tonk', 'house', 'idm', 'indian', 'indie-pop', 'indie',
    'industrial', 'iranian', 'j-dance', 'j-idol', 'j-pop', 'j-rock', 'jazz',
    'k-pop', 'kids', 'latin', 'latino', 'malay', 'mandopop', 'metal',
    'metalcore', 'minimal-techno', 'mpb', 'new-age', 'opera', 'pagode',
    'party', 'piano', 'pop-film', 'pop', 'power-pop', 'progressive-house',
    'psych-rock', 'punk-rock', 'punk', 'r-n-b', 'reggae', 'reggaeton',
    'rock-n-roll', 'rock', 'rockabilly', 'romance', 'sad', 'salsa', 'samba',
    'sertanejo', 'show-tunes', 'singer-songwriter', 'ska', 'sleep',
    'songwriter', 'soul', 'spanish', 'study', 'swedish', 'synth-pop', 'tango',
    'techno', 'trance', 'trip-hop', 'turkish', 'world-music'
]

df['genre'] = df['track_genre'].str.strip().str.lower()

valid_genres = set(full_genre_list)

df['is_valid_genre'] = df['genre'].isin(valid_genres)

df['genre'] = df.apply(
    lambda x: x['genre'] if x['is_valid_genre'] else 'unknown',
    axis=1
)


full_genre_list_with_unknown = ['unknown'] + full_genre_list

label_encoder = LabelEncoder()
label_encoder.fit(full_genre_list_with_unknown)  

df['genre_code'] = label_encoder.transform(df['genre'])

genre_mapping = pd.DataFrame({
    'genre_name': label_encoder.classes_,
    'genre_code': label_encoder.transform(label_encoder.classes_)
}).sort_values('genre_code')

df = df.drop(columns=['is_valid_genre'])

print(df[['track_name', 'genre', 'genre_code']])
df.drop(columns=['genre'], inplace=True)
print(genre_mapping.to_string(index=False))


                        track_name        genre  genre_code
0                           Comedy     acoustic           0
1                 Ghost - Acoustic     acoustic           0
2                   To Begin Again     acoustic           0
3       Can't Help Falling In Love     acoustic           0
4                          Hold On     acoustic           0
...                            ...          ...         ...
113995         Sleep My Little Boy  world-music         114
113996            Water Into Light  world-music         114
113997              Miss Perfumado  world-music         114
113998                     Friends  world-music         114
113999                   Barbincor  world-music         114

[113999 rows x 3 columns]
       genre_name  genre_code
         acoustic           0
         afrobeat           1
         alt-rock           2
      alternative           3
          ambient           4
            anime           5
      black-metal           6
        blueg

In [5]:
df.head()
df.shape


(113999, 19)

In [6]:
df = df.drop_duplicates(subset=["track_name", "artists"])
df.reset_index(drop=True, inplace=True)
print(df.head())
print(df.shape)


                  artists                  track_name  popularity  \
0             Gen Hoshino                      Comedy        0.73   
1            Ben Woodward            Ghost - Acoustic        0.55   
2  Ingrid Michaelson;ZAYN              To Begin Again        0.57   
3            Kina Grannis  Can't Help Falling In Love        0.71   
4        Chord Overstreet                     Hold On        0.82   

   duration_ms  explicit  danceability  energy  key  loudness  mode  \
0       230666         0         0.676  0.4610    1    -6.746     0   
1       149610         0         0.420  0.1660    1   -17.235     1   
2       210826         0         0.438  0.3590    0    -9.734     1   
3       201933         0         0.266  0.0596    0   -18.515     1   
4       198853         0         0.618  0.4430    2    -9.681     1   

   speechiness  acousticness  instrumentalness  liveness  valence     tempo  \
0       0.1430        0.0322          0.000001    0.3580    0.715  0.361245   


In [7]:
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from scipy.spatial import distance


In [ ]:
from sklearn.neighbors import NearestNeighbors
from scipy.spatial import distance

def similar_songs_genre(song_name, song_artist):
    song_index = -1
    find_index = df.loc[(df['track_name'] == song_name) & (df['artists'] == song_artist)]
    if find_index.empty:
        print("Oops, enter a valid artist and song instead!")
        return 
    else:
        song_index = find_index.index[0]

    all_song_names = df["track_name"].to_list()
    all_song_artists = df["artists"].to_list()
    all_song_genres = df["genre_code"].to_list()
    all_song_valence = df["valence"].to_list()
    all_song_danceability = df["danceability"].to_list()

    song_genre = all_song_genres[song_index]
    song_valence = all_song_valence[song_index]
    song_danceability = all_song_danceability[song_index]

    song_distances = []

    for s in range(len(all_song_names)):
        if s != song_index:
            same_artist = all_song_artists[s].strip().lower() == all_song_artists[song_index].strip().lower()
            similar_title = song_name.strip().lower() in all_song_names[s].strip().lower() or \
                            all_song_names[s].strip().lower() in song_name.strip().lower()
            
            if not (same_artist and similar_title):
                if all_song_genres[s] == song_genre:
                    dist = distance.euclidean(
                        (song_valence, song_danceability),
                        (all_song_valence[s], all_song_danceability[s])
                    )
                    similarity = 1 / (1 + dist)
                    song_info = [all_song_names[s], all_song_artists[s], full_genre_list[all_song_genres[s]], similarity]
                    song_distances.append(song_info)

    song_distances.sort(key=lambda x: x[3], reverse=True)

    top_5 = song_distances[:5]
    print(f"\nTop 5 similar songs to '{song_name}' by {song_artist} (genre-matched):")
    for s in top_5:
        print(f"{s[0]} by {s[1]} | Genre: {s[2]} | Similarity Score: {s[3]:.3f}")

    return top_5


def similar_songs_nogenre(song_name, song_artist):
    song_index = -1
    find_index = df.loc[(df['track_name'] == song_name) & (df['artists'] == song_artist)]
    if find_index.empty:
        print("Oops, enter a valid artist and song instead!")
        return 
    else:
        song_index = find_index.index[0]

    all_song_names = df["track_name"].to_list()
    all_song_artists = df["artists"].to_list()
    all_song_valence = df["valence"].to_list()
    all_song_danceability = df["danceability"].to_list()

    song_valence = all_song_valence[song_index]
    song_danceability = all_song_danceability[song_index]

    song_distances = []

    for s in range(len(all_song_names)):
        if s != song_index:
            same_artist = all_song_artists[s].strip().lower() == all_song_artists[song_index].strip().lower()
            similar_title = song_name.strip().lower() in all_song_names[s].strip().lower() or \
                            all_song_names[s].strip().lower() in song_name.strip().lower()

            if not (same_artist and similar_title):
                dist = distance.euclidean(
                    (song_valence, song_danceability),
                    (all_song_valence[s], all_song_danceability[s])
                )
                similarity = 1 / (1 + dist)
                song_info = [all_song_names[s], all_song_artists[s], similarity]
                song_distances.append(song_info)

    song_distances.sort(key=lambda x: x[2], reverse=True)

    top_5 = song_distances[:5]
    print(f"\nTop 5 similar songs to '{song_name}' by {song_artist} (no genre restriction):")
    for s in top_5:
        print(f"{s[0]} by {s[1]} | Similarity Score: {s[2]:.3f}")

    return top_5


print("Examples using genre, valence, and danceability:\n")
similar_songs_genre("Can't Help Falling In Love", "Kina Grannis")
similar_songs_genre("I Ain't Worried", "OneRepublic")
similar_songs_genre("Piano Man", "Billy Joel")
similar_songs_genre("You Belong With Me (Taylor’s Version)", "Taylor Swift")

"""
print("\nExamples using valence and danceability only (no genre):\n")
similar_songs_nogenre("Can't Help Falling In Love", "Kina Grannis")
similar_songs_nogenre("I Ain't Worried", "OneRepublic")
similar_songs_nogenre("Piano Man", "Billy Joel")
similar_songs_nogenre("You Belong With Me (Taylor’s Version)", "Taylor Swift")
"""


Examples using genre, valence, and danceability:


Top 5 similar songs to 'Can't Help Falling In Love' by Kina Grannis (genre-matched):
In the Morning by JJ Heller | Genre: acoustic | Similarity Score: 0.986
Golden Hour - Acoustic Piano by Ben Woodward | Genre: acoustic | Similarity Score: 0.974
Over You by Ingrid Michaelson;A Great Big World | Genre: acoustic | Similarity Score: 0.968
The Power of Love by Gabrielle Aplin | Genre: acoustic | Similarity Score: 0.967
Blackbird Song by Lee DeWyze | Genre: acoustic | Similarity Score: 0.961

Top 5 similar songs to 'I Ain't Worried' by OneRepublic (genre-matched):
Paper Bag by Fiona Apple | Genre: piano | Similarity Score: 0.982
Cloudbusting by Kate Bush | Genre: piano | Similarity Score: 0.973
My Life by Billy Joel | Genre: piano | Similarity Score: 0.959
I Ain’t Worried - Acoustic by OneRepublic | Genre: piano | Similarity Score: 0.959
Hold Me Closer - Purple Disco Machine Remix by Elton John;Britney Spears;Purple Disco Machine | Genre: p

'\nprint("\nExamples using valence and danceability only (no genre):\n")\nsimilar_songs_nogenre("Can\'t Help Falling In Love", "Kina Grannis")\nsimilar_songs_nogenre("I Ain\'t Worried", "OneRepublic")\nsimilar_songs_nogenre("Piano Man", "Billy Joel")\nsimilar_songs_nogenre("You Belong With Me (Taylor’s Version)", "Taylor Swift")\n'